In [1]:
import os
import copy
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
 
import torch
import timm
import torchvision.transforms as transforms
 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    RandomizedSearchCV, train_test_split
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import VotingClassifier
from scipy.stats import loguniform
from xgboost import XGBClassifier
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

c:\Users\MCuartero\Uni\machine-learning\fine-grained-visual-recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration
BASE_DIR = Path("data/animals")

TRAIN_META = BASE_DIR / "train_metadata.csv"
TEST_META = BASE_DIR / "test_metadata.csv"
COLOR_HIST = BASE_DIR / "color_histogram.csv"
HOG_PCA = BASE_DIR / "hog_pca.csv"
ADDITIONAL_FEATS = BASE_DIR / "additional_features.csv"

TRAIN_VIT_CACHE = BASE_DIR / "train_vit_base_embeddings.npy"
TEST_VIT_CACHE  = BASE_DIR / "test_vit_base_embeddings.npy"
TRAIN_RN_CACHE  = BASE_DIR / "train_resnet50_embeddings.npy"
TEST_RN_CACHE   = BASE_DIR / "test_resnet50_embeddings.npy"

SUBMISSION_PATH = BASE_DIR / "submission.csv"

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


In [3]:
# Load CSV Features

print("\n--- Loading CSV features ---")
 
train_meta = pd.read_csv(TRAIN_META)
test_meta  = pd.read_csv(TEST_META)
 
color = pd.read_csv(COLOR_HIST)
hog = pd.read_csv(HOG_PCA)
add = pd.read_csv(ADDITIONAL_FEATS)
 
def merge_features(meta, *feature_dfs):
    df = meta.copy()
    for fdf in feature_dfs:
        df = df.merge(fdf, on="image_id", how="left")
    return df
 
train_df = merge_features(train_meta, color, hog, add)
test_df  = merge_features(test_meta,  color, hog, add)
 
meta_cols = ["image_id", "image_path", "class_id", "class_name"]
csv_feature_cols = [c for c in train_df.columns if c not in meta_cols]
 
print(f"Train samples: {len(train_df)},  Test samples: {len(test_df)}")
print(f"CSV feature dims: {len(csv_feature_cols)}")


--- Loading CSV features ---
Train samples: 3750,  Test samples: 1250
CSV feature dims: 219


In [4]:
def extract_embeddings(image_paths, base_dir, cache_path, model_name, batch_size=32):
    """
    Extract embeddings from any timm model.
    Uses a disk cache so re-running the script is fast.
    """
    if cache_path.exists():
        print(f"  Loading cached embeddings from {cache_path}")
        return np.load(cache_path)
 
    print(f"  Extracting {model_name} embeddings for {len(image_paths)} images ...")
 
    model = timm.create_model(model_name, pretrained=True, num_classes=0)
    model.eval().to(DEVICE)
 
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
 
    all_embeddings = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i : i + batch_size]
        imgs = []
        for rel_path in batch_paths:
            full_path = base_dir / rel_path
            try:
                img = Image.open(full_path).convert("RGB")
            except Exception:
                img = Image.new("RGB", (64, 64), (128, 128, 128))
            imgs.append(preprocess(img))
 
        batch_tensor = torch.stack(imgs).to(DEVICE)
        with torch.no_grad():
            emb = model(batch_tensor).cpu().numpy()
        all_embeddings.append(emb)
 
        if (i // batch_size) % 10 == 0:
            print(f"    {min(i + batch_size, len(image_paths))}/{len(image_paths)}")
 
    embeddings = np.vstack(all_embeddings)
    np.save(cache_path, embeddings)
    print(f"  Saved to {cache_path}")
    return embeddings
 
 
# ── ViT-Base/16: 768-dim, richer than ViT-Small (384-dim) ──────────────────
print("\n--- Extracting ViT-Base embeddings ---")
train_vit = extract_embeddings(
    train_df["image_path"].values, BASE_DIR,
    TRAIN_VIT_CACHE, "vit_base_patch16_224", BATCH_SIZE
)
test_vit = extract_embeddings(
    test_df["image_path"].values, BASE_DIR,
    TEST_VIT_CACHE, "vit_base_patch16_224", BATCH_SIZE
)
 
# ── ResNet-50: 2048-dim, CNN complements ViT's attention-based features ──────
print("\n--- Extracting ResNet-50 embeddings ---")
train_rn = extract_embeddings(
    train_df["image_path"].values, BASE_DIR,
    TRAIN_RN_CACHE, "resnet50", BATCH_SIZE
)
test_rn = extract_embeddings(
    test_df["image_path"].values, BASE_DIR,
    TEST_RN_CACHE, "resnet50", BATCH_SIZE
)
 
print(f"ViT-Base shape:  {train_vit.shape}")
print(f"ResNet-50 shape: {train_rn.shape}")


--- Extracting ViT-Base embeddings ---
  Loading cached embeddings from data\animals\train_vit_base_embeddings.npy
  Loading cached embeddings from data\animals\test_vit_base_embeddings.npy

--- Extracting ResNet-50 embeddings ---
  Loading cached embeddings from data\animals\train_resnet50_embeddings.npy
  Loading cached embeddings from data\animals\test_resnet50_embeddings.npy
ViT-Base shape:  (3750, 768)
ResNet-50 shape: (3750, 2048)


In [5]:
# Combine and normalise features

print("\n--- Combining and normalising features ---")
 
X_train_csv = np.nan_to_num(train_df[csv_feature_cols].values.astype(np.float32))
X_test_csv  = np.nan_to_num(test_df[csv_feature_cols].values.astype(np.float32))
 
scaler_csv = StandardScaler()
X_train_csv = scaler_csv.fit_transform(X_train_csv)
X_test_csv  = scaler_csv.transform(X_test_csv)
 
scaler_vit = StandardScaler()
X_train_vit = scaler_vit.fit_transform(train_vit)
X_test_vit  = scaler_vit.transform(test_vit)
 
scaler_rn = StandardScaler()
X_train_rn = scaler_rn.fit_transform(train_rn)
X_test_rn  = scaler_rn.transform(test_rn)
 
# Concatenate: [ViT-Base | ResNet-50 | CSV features]
X_train = np.hstack([X_train_vit, X_train_rn, X_train_csv])
X_test  = np.hstack([X_test_vit,  X_test_rn,  X_test_csv])
 
y_train = train_df["class_id"].values
 
print(f"Total feature dims: {X_train.shape[1]}")
print(f"  ViT-Base:  {X_train_vit.shape[1]}")
print(f"  ResNet-50: {X_train_rn.shape[1]}")
print(f"  CSV:       {X_train_csv.shape[1]}")


--- Combining and normalising features ---
Total feature dims: 3035
  ViT-Base:  768
  ResNet-50: 2048
  CSV:       219


In [6]:
# Cross validation helper

def cv_evaluate(model, X, y, n_splits=5, label="Model"):
    """Stratified 5-fold cross-validation."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
    print(f"  {label}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {np.round(scores, 4)})")
    return scores.mean()
 

In [7]:
# Model 1: Logistic Regression

print("\n--- Model 1: Logistic Regression ---")
 
lr = LogisticRegression(
    C=10.0,               # Higher C than baseline — richer features warrant less regularisation
    max_iter=2000,
    solver="lbfgs",
    class_weight="balanced",
    random_state=42
)
lr_cv = cv_evaluate(lr, X_train, y_train, label="Logistic Regression")


--- Model 1: Logistic Regression ---
  Logistic Regression: 0.9480 ± 0.0102  (folds: [0.9293 0.9547 0.9533 0.9573 0.9453])


In [8]:
# SVM with RandomizedSearchCV

print("\n--- Model 2: SVM (RBF) with RandomizedSearchCV ---")
 
param_dist = {
    "C": loguniform(1, 1000), # Search log-uniformly in [1, 1000]
    "gamma": loguniform(1e-4, 1e-1), # Search log-uniformly in [0.0001, 0.1]
}
 
svm_search = RandomizedSearchCV(
    SVC(kernel="rbf", probability=True, random_state=42),
    param_distributions=param_dist,
    n_iter=15, # 15 random combos — reasonable on CPU
    cv=3, # 3-fold for speed during the search
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=1
)
svm_search.fit(X_train, y_train)
 
print(f"  Best params: {svm_search.best_params_}")
print(f"  Best CV score: {svm_search.best_score_:.4f}")
 
svm = svm_search.best_estimator_
svm_cv = svm_search.best_score_


--- Model 2: SVM (RBF) with RandomizedSearchCV ---
Fitting 3 folds for each of 15 candidates, totalling 45 fits


KeyboardInterrupt: 

In [ ]:
# Model 3: XGBoost

print("\n--- Model 3: XGBoost ---")
 
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.6,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    device="cpu"
)
xgb_cv = cv_evaluate(xgb, X_train, y_train, label="XGBoost")